# QPUF Job Analysis

Reads the `.txt` result files produced by `retrieve_jobs.py` from the `jobs_results/` directory  
and performs the same two-stage QPE analysis as the original notebook.

**Workflow**
1. Run `retrieve_jobs.py` on the DCV to download results into `jobs_results/`
2. Copy `jobs_results/` to this directory on your local machine
3. Run this notebook

In [ ]:
import json
import os
import glob

import numpy as np
import matplotlib.pyplot as plt

## Helper Functions

In [ ]:
def cyclic_distance(a: int, b: int, n_prec: int) -> int:
    """Cyclic distance |a - b| mod 2^n_prec."""
    M    = 2 ** n_prec
    diff = abs(a - b)
    return min(diff, M - diff)


def parse_outcome(bitstring: str, n_prec: int):
    """
    Parse a Qiskit counts key with two named ClassicalRegisters.
    Key format: 'c2_bits c1_bits'  (space-separated, c2 printed first).
    Returns (m1, m2) as integers.
    """
    parts = bitstring.split(' ')
    return int(parts[1], 2), int(parts[0], 2)


def analyse_counts(counts: dict, n_prec: int, delta: int, label: str = ''):
    """Print acceptance stats and top outcomes for a counts dict."""
    total    = sum(counts.values())
    accepted = sum(
        cnt for outcome, cnt in counts.items()
        if cyclic_distance(*parse_outcome(outcome, n_prec), n_prec) <= delta
    )
    acc_rate = accepted / total

    prefix = f'[{label}] ' if label else ''
    print(f'{prefix}Total shots        : {total}')
    print(f'{prefix}Accepted (dist ≤ {delta}) : {accepted}')
    print(f'{prefix}Acceptance rate    : {acc_rate:.4f}')

    print(f'\n{prefix}Top 10 outcomes:')
    print(f'  {{"bitstring":28s}}  m1  m2  dist  count')
    print(f'  {"-"*52}')
    for k, v in sorted(counts.items(), key=lambda x: -x[1])[:10]:
        m1, m2 = parse_outcome(k, n_prec)
        print(f'  {k!r:28s}  {m1:3d}  {m2:3d}  {cyclic_distance(m1, m2, n_prec):4d}  {v}')

    return total, accepted, acc_rate


def angles_to_unitary(angles: dict) -> np.ndarray:
    phi, theta, lam = angles['phi'], angles['theta'], angles['lam']
    def Rz(a): return np.array([[np.exp(-1j*a/2), 0], [0, np.exp(1j*a/2)]])
    def Ry(a): return np.array([[np.cos(a/2), -np.sin(a/2)], [np.sin(a/2), np.cos(a/2)]])
    return Rz(phi) @ Ry(theta) @ Rz(lam)


def print_eigenvalue_bins(angles: dict, n_prec: int):
    unitary = angles_to_unitary(angles)
    print('Theoretical QPE bins (ideal):')
    for ev in np.linalg.eigvals(unitary):
        phase = np.angle(ev) / (2 * np.pi)
        if phase < 0:
            phase += 1
        print(f'  phase={phase:.4f}  ideal bin={round(phase * 2**n_prec) % 2**n_prec}')


def plot_m1_vs_m2(counts: dict, n_prec: int, n_shots: int, title_suffix: str = ''):
    """Scatter plot of m1 vs m2 phase estimates."""
    m1_vals, m2_vals, weights = [], [], []
    for outcome, count in counts.items():
        m1, m2 = parse_outcome(outcome, n_prec)
        m1_vals.append(m1)
        m2_vals.append(m2)
        weights.append(count)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(m1_vals, m2_vals, s=np.array(weights) * 2.0, alpha=0.6)
    ax.plot([0, 2**n_prec - 1], [0, 2**n_prec - 1], 'r--', label='m1 = m2')
    ax.set_xlabel('m1  (Stage 1 QPE)')
    ax.set_ylabel('m2  (Stage 2 QPE)')
    title = f'Phase estimates – 1-qubit Haar unitary\n(n_prec={n_prec}, n_shots={n_shots})'
    if title_suffix:
        title += f'  {title_suffix}'
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_acceptance_vs_delta(counts: dict, n_prec: int, n_shots: int, delta: int):
    """Plot acceptance rate as a function of the delta threshold."""
    total       = sum(counts.values())
    delta_range = list(range(0, 2**n_prec + 1))
    acc_rates   = []

    for d in delta_range:
        acc = sum(
            cnt for outcome, cnt in counts.items()
            if cyclic_distance(*parse_outcome(outcome, n_prec), n_prec) <= d
        )
        acc_rates.append(acc / total)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(delta_range, acc_rates, marker='o', markersize=4)
    ax.axvline(delta, color='r', linestyle='--', label=f'current delta={delta}')
    ax.set_xlabel('delta  (cyclic distance threshold)')
    ax.set_ylabel('acceptance rate')
    ax.set_title(f'Acceptance rate vs delta  (n_prec={n_prec}, n_shots={n_shots})')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print('\ndelta → acceptance rate')
    for d, r in zip(delta_range, acc_rates):
        print(f'  delta={d:3d}: {r:.4f}')

## Load Result Files

Scans `jobs_results/` for all `.txt` files and lists what was found.

In [ ]:
RESULTS_DIR = os.path.join(os.getcwd(), 'jobs_results')
DELTA       = 2   # cyclic-distance acceptance threshold

txt_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.txt')))

if not txt_files:
    raise FileNotFoundError(
        f'No .txt files found in {RESULTS_DIR}\n'
        'Copy the jobs_results/ directory here after running retrieve_jobs.py on the DCV.'
    )

jobs = []
for path in txt_files:
    with open(path) as f:
        jobs.append(json.load(f))

print(f'Found {len(jobs)} result file(s) in {RESULTS_DIR}\n')
print(f'  {"#":>3}  {"datetime":32s}  {"device":8s}  {"n_prec":7s}  {"n_shots":7s}  task-uuid')
print(f'  {"-"*90}')
for i, job in enumerate(jobs):
    uuid = job['job_id'].split('/')[-1]
    print(f'  {i:>3}  {job["datetime"]:32s}  {job["device"]:8s}  '
          f'{job["n_prec"]:7d}  {job["n_shots"]:7d}  {uuid}')

## Analyse Each Job

For every loaded result file: print acceptance stats, theoretical eigenvalue bins,
and display the scatter + delta-sweep plots.

In [ ]:
summary = []   # collect (uuid, device, n_shots, acceptance_rate) for comparison table

for job in jobs:
    uuid    = job['job_id'].split('/')[-1]
    n_prec  = job['n_prec']
    n_shots = job['n_shots']
    angles  = job['angles']
    counts  = job['counts']
    device  = job['device']

    print('=' * 70)
    print(f'Job      : {uuid}')
    print(f'Submitted: {job["datetime"]}')
    print(f'Device   : {device}  |  n_prec={n_prec}  n_shots={n_shots}')
    print(f'Angles   : phi={angles["phi"]:.6f}  '
          f'theta={angles["theta"]:.6f}  lam={angles["lam"]:.6f}')
    print()

    # ── Acceptance stats ──────────────────────────────────────────────────
    total, accepted, acc_rate = analyse_counts(
        counts, n_prec, DELTA, label=f'{device} hardware'
    )

    print()
    print_eigenvalue_bins(angles, n_prec)

    # ── Plots ─────────────────────────────────────────────────────────────
    plot_m1_vs_m2(counts, n_prec, n_shots,
                  title_suffix=f'[{device}  {uuid[:8]}…]')

    plot_acceptance_vs_delta(counts, n_prec, n_shots, DELTA)

    summary.append((uuid[:8] + '…', device, n_shots, acc_rate))
    print()

## Summary Table

In [ ]:
print('=== Summary ===')
print(f'  {"Task UUID":14s}  {"Device":8s}  {"Shots":>6s}  {"Acceptance rate":>16s}')
print(f'  {"-"*50}')
for uuid_short, device, shots, acc in summary:
    print(f'  {uuid_short:14s}  {device:8s}  {shots:>6d}  {acc:>16.4f}')